# Schuck, Cai, Wilson & Niv (2016) on CRCNS data set ofc-4

Paper: Schuck NW, Cai MB, Wilson RC, Niv Y (2016). Human orbitofrontal cortex represents a cognitive
map of state space. Neuron 91:1402-1412, doi:10.1016/j.neuron.2016.08.019. The claims file
(`claims/crcns-ofc-4/10.1016-j.neuron.2016.08.019.yaml`) was written from the version of record with
its Supplemental Information.

Data set: CRCNS ofc-4, "fMRI time series in orbitofrontal cortex during face-house state-space task"
(Schuck, Cai, Wilson & Niv 2019), doi:10.6080/K0FN14CH, read from the public S3 bucket
`crcnsarchive` under `ofc-4/live/`. Version: `unversioned`. CRCNS data sets carry no version numbers
(docs/archives.md); the bucket holds one copy, under `live/` (no `old/` folder), described by version
0.6 of the data descriptor (April 19, 2019), the only version. The paper cites no version of the data
set, which was published three years after it, so this copy is the one to pin. The notebook prints
the data set's `checksums.md5` and the MD5 of the zip as streamed; the files read are `FHSS_OFC.zip`
(MD5 4372953c3900c5a965ad09526a9510ed) and `checksums.md5`, so a later change of the zip is detectable.

The data set holds, for the 24 participants of the paper's trial-wise analysis (the paper analysed
27), the OFC time series in native space with no voxel coordinates, the design matrices (16 state
columns, 'first', 'err') with drift and motion nuisance time courses, and the event files (onset,
reaction time, stimulus duration per state, 'first' and 'err'). It holds no whole-brain data, no
other ROI, no training session, no response side or response mapping, and no voxel coordinates.

| Claim | Tested here |
|---|---|
| behavior-error-rate | tested |
| supp-errors-by-trial-class | tested |
| supp-errors-stable-over-blocks | tested |
| supp-errors-not-by-age-category | tested |
| supp-training-errors-higher | blocked: data_missing (no training session) |
| supp-training-rt-unchanged | blocked: data_missing (no training session) |
| supp-rt-stable-over-blocks | tested |
| supp-rt-faces-faster | tested |
| supp-rt-by-trial-class | tested |
| fig2a-16way-decoding | tested |
| fig2-two-back-state-space | tested |
| fig2b-hidden-components | tested |
| fig2b-current-age-absent | tested |
| fig2b-two-back-absent | tested |
| figs2c-8way-decoding | tested |
| figs2a-permutation-control | tested |
| synthetic-data-control | blocked: metadata_insufficient (no voxel coordinates) |
| fig2c-enter-cluster | tested |
| fig2d-enter-nonenter-decoding | tested |
| fig3-searchlight-conjunction | blocked: data_missing (no whole-brain data) |
| other-rois-incomplete | blocked: data_missing (no other ROIs) |
| fig4a-decoding-error-correlation | tested |
| fig4b-error-trial-drop | tested |
| fig4b-decline-before-errors | tested |
| fig4b-error-vs-matched | tested |
| fig5-similarity-predicts-errors | tested |
| figs5-euclidean-distance | tested |
| fig5-synthetic-control | blocked: metadata_insufficient (depends on synthetic-data-control) |
| supp-motor-action-not-in-ofc | blocked: data_missing (no response side; no whole-brain data) |

Method details taken from the paper, readings and assumptions are given in each section and in
`ofc4.py`. One assumption runs through the decoding claims: the beta maps are z-scored per voxel
across the 16 state maps of a run ('voxel'), the operation of the Princeton MVPA toolbox's
`zscore_runs`, which the Methods name ("Data Preprocessing"); the other reading of O2, each map
across voxels ('map'), is shown everywhere beside it. The notebook was revised after the first
verification round: the readings of O3 (post-error trials), a composition of the error blocks, a
class-matched comparison of error trials, a control for trial counts in the distance analysis and
a by-participant slope variant of the RT model were added. No map or volume is smoothed (O1): the files
hold no voxel coordinates.

In [ ]:
import hashlib
import time
import warnings

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import ofc4
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Mixed-model fits print whether they converged; the optimiser's own warnings are silenced.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
T0 = time.time()
DATASET = "crcns:ofc-4/unversioned"  # doi:10.6080/K0FN14CH

In [ ]:
# Files and checksums of the data set, and the zip's MD5 computed from the stream.
print(ofc4.read_text(f"{ofc4.BASE}/checksums.md5"))
md5 = hashlib.md5()
with fsspec.open(ofc4.ZIP_URL, "rb", block_size=2**22) as f:
    while chunk := f.read(2**22):
        md5.update(chunk)
print("MD5 of FHSS_OFC.zip as streamed:", md5.hexdigest())
names = ofc4.members()
print(len(names), "zip members")
SUBJ = ofc4.subjects()
print("participants:", len(SUBJ), SUBJ)
print("runs per participant:", {s: len(ofc4.runs(s)) for s in SUBJ})

The zip's MD5 as streamed equals the one in `checksums.md5`. The data set holds 24 participants, one
with four runs and 23 with five, as the descriptor says.

In [ ]:
# Every field of every run file: shapes, labels, TR, and checks of the stated contents.
rows = []
for s in SUBJ:
    for r in ofc4.runs(s):
        d = ofc4.load_run(s, r)
        rows.append({"subject": s, "run": r, "n_TR": d["n_TR"], "Y_rows": d["Y"].shape[0], "n_voxel": d["n_voxel"],
                     "Y_cols": d["Y"].shape[1], "X_cols": d["X"].shape[1], "TR": d["TR"],
                     "labels": ",".join(l for l in d["labels"] if l not in ofc4.STATES),
                     "nuis_cols": d["nuis"].shape[1], "nuis0_const": bool(np.allclose(d["nuis"][:, 0], 1)),
                     "Y_min": d["Y"].min(), "Y_max": d["Y"].max(), "Y_mean": d["Y"].mean(),
                     "n_zero_vox": int((d["Y"].std(0) == 0).sum()),
                     "states_present": sum(l in ofc4.STATES for l in d["labels"]), "keys": ",".join(d["keys"]),
                     "md5_Y": hashlib.md5(d["Y"].tobytes()).hexdigest(),
                     "md5_Y_head": hashlib.md5(np.round(d["Y"][:20, :50], 3).tobytes()).hexdigest()})
files = pd.DataFrame(rows)
print("fields of the run files (all variables in each .mat file):", files["keys"].value_counts().to_dict())
print(files.drop(columns=["md5_Y", "md5_Y_head", "keys"]).describe().round(2).to_string())
print("non-state columns:", files.labels.value_counts().to_dict())
print("TR values:", files.TR.unique(), "; n_TR equals rows:", (files.n_TR == files.Y_rows).all(),
      "; n_voxel equals columns:", (files.n_voxel == files.Y_cols).all())
print("distinct time series:", files.md5_Y.nunique(), "of", len(files), "; distinct first 20 x 50 blocks:", files.md5_Y_head.nunique())
print("voxels constant across runs within participant:", files.groupby("subject").n_voxel.nunique().eq(1).all())
print("voxels per participant: min", files.n_voxel.min(), "max", files.n_voxel.max())
print("all 16 states in every run:", (files.states_present == 16).all(), "; first nuisance column constant:", files.nuis0_const.all())
print("runs without a 'first' column:", files.loc[~files.labels.str.contains("first"), ["subject", "run"]].values.tolist())
print("runs without an 'err' column:", files.loc[~files.labels.str.contains("err"), ["subject", "run"]].values.tolist())

Checks of the descriptor against the files. Every run file holds the same seven variables
(`OFC_timecourse`, `TR`, `condition_label`, `design_matrix_task`, `n_TR`, `n_voxel`,
`nuisance_timecourse`) and no mask or voxel coordinates. TR is 2.4 s in every run; `n_TR` and `n_voxel` match the
array shapes; each participant's voxel count is the same in every run (3450 to 4635 voxels), consistent
with one native-space mask per participant, though the files give no coordinates to show it. The 119
time series are all distinct, so no run was deposited twice. All 16 state columns are present in every
run. The descriptor says the 'err' column is missing where a run had no error (31 runs here, all
without error events); it does not say that the 'first' column is missing in three runs (Subj-11 run 3,
Subj-18 runs 4 and 5). The first nuisance column is constant, so the GLM's run mean is in the nuisance
set. Intensities are raw scanner values (278 to 1954).

In [ ]:
# Events: one row per labelled trial, 'first' trial and error block.
EV = pd.concat([ofc4.events(s, r) for s in SUBJ for r in ofc4.runs(s)], ignore_index=True)
EV["is_state"] = EV.cond.isin(ofc4.STATES)
per_run = EV.groupby(["subject", "run"]).agg(labelled=("is_state", "sum"),
                                              first=("cond", lambda c: (c == "first").sum()),
                                              err_blocks=("cond", lambda c: (c == "err").sum()),
                                              last_onset=("onset", "max")).reset_index()
print(per_run.describe().round(2).to_string())
print("RT range (s):", EV.loc[EV.is_state, "rt"].min(), EV.loc[EV.is_state, "rt"].max(),
      "; RT values are multiples of 0.1 s:", np.allclose(EV.rt * 10, np.round(EV.rt * 10)))
print("stimulus duration range (s):", EV.loc[EV.is_state, "dur"].min(), EV.loc[EV.is_state, "dur"].max())
print("onsets equal in RT and dur files:", np.allclose(EV.onset, EV.onset_dur))
# Labelled trials per run plus trials taken up by error blocks: a run of the design has 96 trials after the first.
per_run["designed_in_blocks"] = 96 - per_run.labelled
print("runs with no error block and 96 labelled trials:", ((per_run.err_blocks == 0) & (per_run.labelled == 96)).sum(),
      "of", (per_run.err_blocks == 0).sum(), "runs with no error block")
# Consecutive labelled trials must chain: the previous components of a trial are the current ones of the trial before.
bad = []
for (s, r), g in EV.groupby(["subject", "run"]):
    c = g.cond.tolist()
    for a, b in zip(c, c[1:]):
        if a in ofc4.STATES and b in ofc4.STATES and b[2:] != a[:2]:
            bad.append((s, r, a, b))
print("adjacent labelled trials that do not chain:", bad)
cls = EV[EV.is_state].assign(klass=lambda d: d.cond.map(ofc4.trial_class))
print("labelled trials per class and run (design: 32 Enter, 32 Internal, 32 Exit per run of 96):")
print(cls.groupby(["subject", "run"]).klass.value_counts().unstack().describe().round(1).to_string())

Every run without an error block has 96 labelled trials after the first, the 96 of the design
(Methods, Design), and labelled trials per class are close to the 32 Enter, 32 Internal and 32 Exit
per run the design implies. Labels read as current then previous trial chain in every pair of adjacent
labelled trials but one: in Subj-01 run 4 a 'HoHy' trial follows a 'HoHy' trial, which the task rules
do not allow. RTs are given to 0.1 s (0.4 to 2.7 s), and stimulus durations run from 2.8 to 5.0 s,
inside the 2.75 to 5 s of the Methods.

In [ ]:
# Error blocks: span from onset to the end of the last stimulus, by the class of the trial before.
ERR = pd.concat([ofc4.error_trials(g) for _, g in EV.groupby(["subject", "run"])], ignore_index=True)
ERR["after_exit"] = ERR.P.map(lambda x: x in ofc4.STATES and ofc4.trial_class(x) == "Exit")
print("error blocks:", len(ERR), "; after an Exit trial:", int(ERR.after_exit.sum()), "; on a run's first trial:", int(ERR.P.isna().sum()))
print("span (s) after an Exit trial:", ERR.loc[ERR.after_exit, "span"].describe().round(1).to_dict())
print("span (s) after another trial:", ERR.loc[~ERR.after_exit, "span"].describe().round(1).to_dict())
print("blocks in runs whose labelled trials and blocks add up to 96:", int(ERR.one_per_block.sum()),
      "; of these after an Exit trial:", int((ERR.one_per_block & ERR.after_exit).sum()),
      "; reconstructed states legal:", ERR.legal.value_counts().to_dict())
print("error trials with a state, strict reading:", int(ERR.state.notna().sum()), "; span reading:", int(ERR.state_span.notna().sum()),
      "; the two disagree on:", int((ERR.state.notna() & ERR.state_span.notna() & (ERR.state != ERR.state_span)).sum()))
print("class known (Enter after Exit, or reconstructed):", int(ERR.klass.notna().sum()), "of", len(ERR))
print("runs whose labelled trials and blocks add up to 96, among runs with blocks:",
      int((per_run.err_blocks.gt(0) & (per_run.labelled + per_run.err_blocks == 96)).sum()), "of", int(per_run.err_blocks.gt(0).sum()))
# designed trials taken up per block, in runs whose blocks all follow an Exit trial versus others
per_run2 = per_run.merge(ERR.groupby(["subject", "run"]).after_exit.all().rename("all_after_exit").reset_index(), on=["subject", "run"])
per_run2["extra_per_block"] = (96 - per_run2.labelled - per_run2.err_blocks) / per_run2.err_blocks
print("designed trials beyond one per block, per block: runs with only Enter errors",
      per_run2.loc[per_run2.all_after_exit, "extra_per_block"].mean().round(2),
      "; runs with other errors", per_run2.loc[~per_run2.all_after_exit, "extra_per_block"].mean().round(2))
# Composition of the blocks. Trial slots per block: time from the block's onset to the next event,
# over the run's mean interval between consecutive labelled trials.
slots = []
for (s, r), g in EV.groupby(["subject", "run"]):
    g = g.reset_index(drop=True)
    iti = np.mean([g.onset[i] - g.onset[i - 1] for i in range(1, len(g)) if g.is_state[i] and (g.is_state[i - 1] or g.cond[i - 1] == "first")])
    for i in np.flatnonzero(g.cond.values == "err"):
        P = g.cond[i - 1] if i > 0 else None
        slots.append({"before": ofc4.trial_class(P) if P in ofc4.STATES else "none", "rt": g.rt[i], "span": g.dur[i],
                      "slots": (g.onset[i + 1] - g.onset[i]) / iti if i + 1 < len(g) else np.nan})
slots = pd.DataFrame(slots)
print("trial slots per block (onset to next event over the mean trial interval), by the class of the trial before the block:")
print(slots.groupby("before").slots.describe().round(2).to_string())
print("rounded slots per block:", slots.groupby("before").slots.apply(lambda x: x.dropna().round().astype(int).value_counts().sort_index().to_dict()).dropna().to_dict())
print("'err' durations: RT file", slots.rt.min(), "to", slots.rt.max(), "s; dur file (span)", slots.span.min(), "to", slots.span.max(),
      "s; span less RT duration, median", round((slots.span - slots.rt).median(), 2), "s")
# Designed trials inside the blocks, per block, by least squares over runs: 96 less the labelled trials
# against the counts of blocks after an Exit trial and of other blocks.
cnt = ERR.assign(k=np.where(ERR.after_exit, "after_exit", "other")).groupby(["subject", "run", "k"]).size().unstack(fill_value=0)
cnt = cnt.join(per_run.set_index(["subject", "run"]).labelled)
coef = np.linalg.lstsq(cnt[["after_exit", "other"]].values, 96 - cnt.labelled.values, rcond=None)[0]
print(f"designed trials per block, least squares over {len(cnt)} runs with blocks: after an Exit trial {coef[0]:.2f}, other blocks {coef[1]:.2f}")
print("share of blocks after an Exit trial:", round(ERR.after_exit.mean(), 3),
      "; Enter share of errors implied by the paper's class rates (3.4, 2.3, 1.4% over equal trial counts):", round(3.4 / (3.4 + 2.3 + 1.4), 3))
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist([ERR.loc[ERR.after_exit, "span"], ERR.loc[~ERR.after_exit, "span"]], bins=np.arange(5, 46, 1),
        stacked=True, label=["after Exit", "after Enter or Internal"])
ax.set_xlabel("error block span (s)"); ax.set_ylabel("blocks"); ax.legend()
print("span range plotted 5 to 46 s; outside:", int(((ERR.span < 5) | (ERR.span >= 46)).sum()))
plt.show()

The 'err' events are blocks, not single trials: their spans (onset to end of the last stimulus) run
from 6.8 to 41 s, two to several trials. All 181 blocks that follow an Exit trial hold a trial of the
other category, so their first trial is an Enter error, and the short ones (about two trials) hold the
error and its repetition. In runs whose labelled trials and blocks add up to 96 (27 of 88 runs with
blocks), every block holds one designed trial and the error trial's state follows from its neighbours;
all 39 such states are legal, and they are all Enter errors. Elsewhere, blocks take up more designed
trials than one (0.38 more per block in runs with only Enter errors, 0.69 in others), so a block can
hold more than one error or the trials after it. The number of errors and the state of most error
trials therefore cannot be read from the files: the class is known for 181 of 228 blocks, the full
state for 39 (strict reading) or 98 (span reading).

Composition of the blocks. Measured in trial slots (onset to the next event over the run's mean
trial interval), 82 of the 172 blocks after an Exit trial span about two slots, the error and its
repetition, and most others three to five; blocks after Internal or Enter trials span about four
(median 4.0 and 4.4), as an Exit error followed by the repeated preceding trial and the repeated
Exit trial would. Over runs, each block after an Exit trial takes up 1.52 designed trials and each
other block 1.87 (least squares), so a block after an Exit trial holds on average about half a designed
trial (0.52) besides its error trial, and another block close to one (0.87). Whether that trial is a second error or a correct trial after the
error the files do not say. The 'err' RT-file duration runs to the last response of the block (span
less RT duration, median 2.1 s). Blocks after an Exit trial are 79% of all blocks, where the paper's
class rates imply that Enter errors are 48% of errors: Internal and Exit errors appear undercounted
here, or merged into blocks.

In [ ]:
# Timing: the data set's state columns against regressors rebuilt from the event files (SPM HRF),
# for several shifts of the event times; the shift with the best match is the one the helper uses.
chk = []
for s in SUBJ[:6]:
    for r in ofc4.runs(s)[:2]:
        d = ofc4.load_run(s, r)
        ev = EV[(EV.subject == s) & (EV.run == r)]
        for shift in (-1.2, -0.6, 0.0, 0.6):
            for j, lab in enumerate(d["labels"]):
                sel = ev[ev.cond == lab]
                if len(sel) == 0:
                    continue
                x = ofc4.regressor(sel.onset, sel.rt, d["n_TR"], shift=shift)
                chk.append({"subject": s, "run": r, "shift": shift, "col": lab, "r": np.corrcoef(x, d["X"][:, j])[0, 1]})
chk = pd.DataFrame(chk)
chk["kind"] = np.where(chk.col.isin(ofc4.STATES), "state", chk.col)
print(chk.groupby(["kind", "shift"]).r.agg(["median", "min"]).round(3).to_string())
# The 'err' column against the err durations of the dur files too
errchk = []
for s in SUBJ[:6]:
    for r in ofc4.runs(s)[:2]:
        d = ofc4.load_run(s, r)
        if "err" not in d["labels"]:
            continue
        ev = EV[(EV.subject == s) & (EV.run == r) & (EV.cond == "err")]
        j = d["labels"].index("err")
        errchk.append({"subject": s, "run": r,
                       "r_rt": np.corrcoef(ofc4.regressor(ev.onset, ev.rt, d["n_TR"]), d["X"][:, j])[0, 1],
                       "r_dur": np.corrcoef(ofc4.regressor(ev.onset, ev.dur, d["n_TR"]), d["X"][:, j])[0, 1],
                       "r_impulse": np.corrcoef(ofc4.regressor(ev.onset, np.full(len(ev), 0.1), d["n_TR"]), d["X"][:, j])[0, 1],
                       "err_col_max": d["X"][:, j].max(), "state_col_max": d["X"][:, :16].max()})
print(pd.DataFrame(errchk).round(3).to_string())

The data set's state and 'first' columns match regressors rebuilt from the RT files with SPM's
canonical HRF (median r 0.997 at a shift of -0.6 s, minimum 0.988), so the design matrices and event
files share one clock. The 'err' column does not match the error blocks' RT or stimulus durations
(r 0.30 to 0.76) but matches a short event at each block onset (r 0.83 to 0.98): it is not the boxcar
from stimulus onset to button press that the descriptor describes for the design matrices.

In [ ]:
# Behavioural tables. Designed trials: each labelled trial (correct) and each error block's error
# trial (one error per block, a lower bound when a block holds more). The first trial of a run is
# kept apart. Runs per participant from the files.
NRUN = EV.groupby("subject").run.nunique()
lab = EV[EV.is_state].copy()
lab["klass"] = lab.cond.map(ofc4.trial_class)
lab["cur_cat"] = lab.cond.str[0]
lab["cur_age"] = lab.cond.str[1]
lab["error"] = 0
errt = ERR.copy()
errt["error"] = 1
errt["first_trial"] = errt.P.isna()
trials = pd.concat([lab[["subject", "run", "onset", "klass", "cur_cat", "error"]].assign(cur_age=lab.cur_age, state=lab.cond),
                    errt.loc[~errt.first_trial, ["subject", "run", "onset", "klass", "cur_cat", "error", "state", "state_span"]]
                    .assign(cur_age=lambda d: d.state_span.str[1])], ignore_index=True)
print("designed trials after the first:", len(trials), "; errors:", trials.error.sum(),
      "; error blocks on a run's first trial:", int(errt.first_trial.sum()))
print("errors with class known:", trials.loc[trials.error == 1, "klass"].notna().sum(),
      "; with current age known (span reading):", trials.loc[trials.error == 1, "cur_age"].notna().sum())


def wald(fit, names):
    """Wald chi-square of the named coefficients of a fitted statsmodels model."""
    b = fit.params[names].values
    V = fit.cov_params().loc[names, names].values
    chi2 = float(b @ np.linalg.solve(V, b))
    return chi2, len(names), float(stats.chi2.sf(chi2, len(names)))

## behavior-error-rate

> Participants performed the task nearly without errors, with a mean error rate of 2.3% in the scanner.

Errors are the error blocks of the event files ('err'); the files do not tell time-outs from wrong
responses, so the two readings of the criterion (errors only, errors plus time-outs) cannot be
separated, and both are this count. A block that holds more than one error counts once.
Denominators: (a) all trials including the repetitions that follow errors (97 per run plus one
repetition per error, Methods, Task), (b) all trials but the first of each run (96 per run,
with errors on a first trial left out).

In [ ]:
nerr_all = ERR.groupby("subject").size().reindex(SUBJ, fill_value=0)
nerr_nofirst = ERR[ERR.P.notna()].groupby("subject").size().reindex(SUBJ, fill_value=0)
rate_a = 100 * nerr_all / (97 * NRUN + nerr_all)
rate_b = 100 * nerr_nofirst / (96 * NRUN)
er = pd.DataFrame({"runs": NRUN, "error_blocks": nerr_all, "rate_a": rate_a, "rate_b": rate_b})
print(er.round(2).to_string())
for k in ("rate_a", "rate_b"):
    print(f"{k}: mean {er[k].mean():.2f}% (SEM {er[k].sem():.2f}), median {er[k].median():.2f}, range {er[k].min():.2f} to {er[k].max():.2f}; "
          f"expected 2.3 +/- 0.23")
print("bonus threshold 2.5%: participants below it (reading b):", int((er.rate_b < 2.5).sum()), "of", len(er))
# A block can hold more than one error: the designed trials it takes up (96 per run less the labelled
# trials) bound the number of errors from above, if every designed trial inside a block was an error.
up = per_run.assign(designed_in_blocks=96 - per_run.labelled).groupby("subject").designed_in_blocks.sum().reindex(SUBJ)
er["rate_upper"] = 100 * up / (96 * NRUN)
print(f"upper bound, reading b denominator: mean {er.rate_upper.mean():.2f}% (designed trials inside blocks: {int(up.sum())}, blocks: {int(nerr_all.sum())})")
ERRRATE = er.rate_b / 100

**Result: blocked, metadata insufficient.** The files do not record the number of errors. Counting one
error per block gives a mean of 1.92% (reading a) or 1.98% (reading b), below the 2.07 to 2.53% of the
tolerance; counting every designed trial inside a block as an error gives an upper bound of 3.10%,
above it. The paper's 2.3% lies between these bounds. Time-outs cannot be told from errors, so
the two readings of the criterion give the same count.

## supp-errors-by-trial-class

> Error rates differed by trial class (Enter 3.4%, Exit 2.3%, Internal 1.4%).

An error trial's class is Enter when the trial before its block is an Exit trial. Otherwise it is
Internal or Exit, and the data tell which only where the state is reconstructed (`ofc4.error_trials`).
The errors of unknown class are shown two ways, all Internal and all Exit, which bound each rate. Per participant, the
rate in a class is its errors over its designed trials. Model readings: (a) linear mixed model of
per-participant class rates, random intercept; (b) logistic model of trial-wise errors; statsmodels
has no frequentist logistic mixed model, so (b) uses participant fixed effects in its place.

In [ ]:
unk = trials[(trials.error == 1) & trials.klass.isna()]
print("errors of unknown class (Internal or Exit):", len(unk), "of", int(trials.error.sum()))
cls_res = {}
for assign in ("Internal", "Exit"):
    t = trials.copy()
    t.loc[t.klass.isna(), "klass"] = assign
    pr = t.groupby(["subject", "klass"]).error.mean().mul(100).rename("rate").reset_index()
    means = pr.groupby("klass").rate.mean()
    m = smf.mixedlm("rate ~ C(klass, Treatment('Internal'))", pr, groups=pr["subject"]).fit()
    names = [n for n in m.params.index if n.startswith("C(klass")]
    chi2, df, p = wald(m, names)
    lg = smf.logit("error ~ C(klass, Treatment('Internal')) + C(subject)", t).fit(disp=0)
    chi2b, dfb, pb = wald(lg, [n for n in lg.params.index if n.startswith("C(klass")])
    cls_res[assign] = means
    print(f"unknown errors {assign}: rates Enter {means['Enter']:.2f}%, Exit {means['Exit']:.2f}%, Internal {means['Internal']:.2f}% "
          f"| (a) LMM chi2({df}) = {chi2:.1f}, p = {p:.2g} | (b) logit chi2({dfb}) = {chi2b:.1f}, p = {pb:.2g}")
print("expected: Enter 3.4 +/- 0.34, Exit 2.3 +/- 0.23, Internal 1.4 +/- 0.14")
# Enter against the other two classes together, which the data do tell apart
t = trials.assign(enter=np.where(trials.klass.eq("Enter"), "Enter", "other"))
pr = t.groupby(["subject", "enter"]).error.mean().mul(100).rename("rate").reset_index()
m = smf.mixedlm("rate ~ C(enter)", pr, groups=pr["subject"]).fit()
chi2, df, p = wald(m, ["C(enter)[T.other]"])
print(f"Enter {pr[pr.enter=='Enter'].rate.mean():.2f}% against Internal and Exit together {pr[pr.enter=='other'].rate.mean():.2f}%: "
      f"chi2(1) = {chi2:.1f}, p = {p:.2g}; participants with a higher Enter rate: "
      f"{int((pr.pivot(index='subject', columns='enter', values='rate').Enter > pr.pivot(index='subject', columns='enter', values='rate').other).sum())} of {pr.subject.nunique()}")
print("errors per class, all participants: Enter", int(trials[(trials.error == 1) & trials.klass.eq("Enter")].shape[0]),
      "; Internal or Exit", len(unk), "; trials per class:", trials.klass.value_counts().to_dict())

**Result: partially reproduced, resting on one error per block and the class of an error taken from
the trial before its block.** Error rates differ by class (LMM chi2(2) = 63.6 to 63.8, p < 1e-13, under
both assignments of the 44 errors of unknown class), and the Enter rate is higher than the other two
classes together in all 24 participants. The rates do not match: Enter is 4.79% (expected 3.4 +/- 0.34),
and Exit and Internal together are 0.60%. Given all 44 errors of unknown class, Exit would reach
1.18% or Internal 1.20%, so neither can reach its expected value (2.3% and 1.4%) whichever class the
unknown errors belong to, and whether Exit is above Internal cannot be told.
Blocks after an Exit trial are 79% of blocks against the 48% the paper's rates imply (data check
above), so Internal and Exit errors may be undercounted or merged into blocks.

## supp-errors-stable-over-blocks

> Error rates did not change over the blocks of the main experiment.

Per participant and run, errors over the run's 96 trials after the first; run as a linear term,
random intercept and slope by participant (reading a); (b) trial-wise logistic model with
participant fixed effects. Judged on (a) at p < 0.05.

In [ ]:
rr = trials.groupby(["subject", "run"]).error.mean().mul(100).rename("rate").reset_index()
m = smf.mixedlm("rate ~ run", rr, groups=rr["subject"], re_formula="~run").fit()
chi2, df, p = wald(m, ["run"])
print(f"(a) LMM: slope {m.params['run']:.3f} percentage points per run, chi2({df}) = {chi2:.2f}, p = {p:.3f}; converged {m.converged}")
lg = smf.logit("error ~ run + C(subject)", trials).fit(disp=0)
chi2b, dfb, pb = wald(lg, ["run"])
print(f"(b) logit: log-odds slope {lg.params['run']:.3f} per run, chi2({dfb}) = {chi2b:.2f}, p = {pb:.3f}")
print("mean error rate per run (%):", rr.groupby("run").rate.mean().round(2).to_dict())
print("paper: chi2(1) = 0.08, p = .77")

**Result: reproduced, resting on one error per block.** The run effect on per-run error rates is not significant (slope -0.11
percentage points per run, chi2(1) = 1.44, p = 0.23; logistic variant p = 0.22). Errors are counted one
per block.

## supp-errors-not-by-age-category

> Error rates were not affected by the age or the category judged, nor by their interaction.

The current category of every error trial follows from the trial before its block; its current age
is known only for reconstructed Enter errors (span reading, `state_span`). The full model of the
criterion (category, age, interaction; random intercept) therefore runs on Enter trials whose age is
known (labelled Enter trials and the reconstructed Enter errors), which leaves out the other errors;
the category alone is also tested on all trials and errors.

In [ ]:
ta = trials[trials.klass.eq("Enter") & trials.cur_age.notna()]
pa = ta.groupby(["subject", "cur_cat", "cur_age"]).error.mean().mul(100).rename("rate").reset_index()
m = smf.mixedlm("rate ~ C(cur_cat) * C(cur_age)", pa, groups=pa["subject"]).fit()
for term in ["C(cur_cat)[T.H]", "C(cur_age)[T.y]", "C(cur_cat)[T.H]:C(cur_age)[T.y]"]:
    chi2, df, p = wald(m, [term])
    print(f"(a) LMM, Enter trials with known age ({int(ta.error.sum())} errors): {term}: coef {m.params[term]:.3f}, chi2 = {chi2:.2f}, p = {p:.3f}")
lg = smf.logit("error ~ C(cur_cat) * C(cur_age) + C(subject)", ta).fit(disp=0)
for term in ["C(cur_cat)[T.H]", "C(cur_age)[T.y]", "C(cur_cat)[T.H]:C(cur_age)[T.y]"]:
    chi2, df, p = wald(lg, [term])
    print(f"(b) logit: {term}: chi2 = {chi2:.2f}, p = {p:.3f}")
tc = trials[trials.cur_cat.notna()]
pc = tc.groupby(["subject", "cur_cat"]).error.mean().mul(100).rename("rate").reset_index()
m = smf.mixedlm("rate ~ C(cur_cat)", pc, groups=pc["subject"]).fit()
chi2, df, p = wald(m, ["C(cur_cat)[T.H]"])
print(f"category alone, all {int(tc.error.sum())} errors: faces {pc[pc.cur_cat=='F'].rate.mean():.2f}%, houses {pc[pc.cur_cat=='H'].rate.mean():.2f}%, "
      f"chi2 = {chi2:.2f}, p = {p:.3f}")
print("cell rates (errors with known age), mean over participants (%):")
print(pa.groupby(["cur_cat", "cur_age"]).rate.mean().round(2).to_string())

**Result: reproduced, resting on the span reading of error states.** On Enter trials whose age is known,
category (p = 0.41), age (p = 0.63) and their interaction (p = 0.95) do not affect error rates, and the
logistic variant agrees (p = 0.44, 0.61, 0.88). The category alone, over all trials, has p = 0.23. The
age test covers only the 98 reconstructed Enter errors.

## supp-rt-stable-over-blocks, supp-rt-faces-faster, supp-rt-by-trial-class

Reaction times are the durations of the RT event files, given to 0.1 s, for the labelled (correct)
trials after the first. Error trials have no RT of their own in the files (an error block's
duration is its span), so the reading "all responded trials" cannot be run; only correct trials
are used. RTs are medians per participant and factor cell (Methods, Behavioral Analyses).

In [ ]:
lab["rt_ms"] = lab.rt * 1000
rb = lab.groupby(["subject", "run"]).rt_ms.median().rename("rt").reset_index()
m = smf.mixedlm("rt ~ run", rb, groups=rb["subject"], re_formula="~run").fit()
chi2, df, p = wald(m, ["run"])
print(f"LMM: slope {m.params['run']:.2f} ms per run, chi2({df}) = {chi2:.2f}, p = {p:.3f}; converged {m.converged}")
print("mean of participant medians per run (ms):", rb.groupby("run").rt.mean().round(1).to_dict())
print("paper: chi2(1) = 1.7, p = .18")

**Result: reproduced.** RT medians do not change over runs (slope -8.6 ms per run, chi2(1) = 1.74,
p = 0.19; the paper gives chi2(1) = 1.7, p = .18).

In [ ]:
rc = lab.groupby(["subject", "cur_cat", "cur_age"]).rt_ms.median().rename("rt").reset_index()
m = smf.mixedlm("rt ~ C(cur_cat) * C(cur_age)", rc, groups=rc["subject"]).fit()
for term in ["C(cur_cat)[T.H]", "C(cur_age)[T.y]", "C(cur_cat)[T.H]:C(cur_age)[T.y]"]:
    chi2, df, p = wald(m, [term])
    print(f"LMM {term}: coef {m.params[term]:.1f} ms, chi2(1) = {chi2:.1f}, p = {p:.3g}")
# variant: random slopes of category by participant, which the criterion's random-intercept model leaves out
m2 = smf.mixedlm("rt ~ C(cur_cat) * C(cur_age)", rc, groups=rc["subject"], re_formula="~C(cur_cat)").fit()
chi2, df, p = wald(m2, ["C(cur_cat)[T.H]"])
print(f"variant with by-participant category slopes: category coef {m2.params['C(cur_cat)[T.H]']:.1f} ms, chi2(1) = {chi2:.1f}, p = {p:.3g}; converged {m2.converged}")
cat_med = lab.groupby(["subject", "cur_cat"]).rt_ms.median().unstack()
print("mean over participants of category medians (ms): faces %.1f, houses %.1f" % tuple(cat_med[["F", "H"]].mean()))
print("median of pooled trials (ms): faces %.1f, houses %.1f" % (lab[lab.cur_cat == "F"].rt_ms.median(), lab[lab.cur_cat == "H"].rt_ms.median()))
print("participants faster for faces:", int((cat_med.F < cat_med.H).sum()), "of", len(cat_med),
      "; expected faces 944 +/- 94.4, houses 1028 +/- 102.8")

**Result: reproduced.** Faces are faster than houses (mean of participant medians 945.8 against
1012.5 ms, both within tolerance; LMM category chi2(1) = 58.7, p = 1.8e-14), with no effect of age
(p = 0.72) and no interaction (p = 0.61). With by-participant category slopes added (a variant; the
criterion's model has a random intercept only), category chi2(1) = 54.1, p = 1.9e-13. Pooled trial medians are 900 and 1000 ms. RTs are given to
0.1 s, so many participant medians tie: 15 of 24 participants have a lower face median.

In [ ]:
rk = lab.groupby(["subject", "klass"]).rt_ms.median().rename("rt").reset_index()
m = smf.mixedlm("rt ~ C(klass, Treatment('Internal'))", rk, groups=rk["subject"]).fit()
chi2, df, p = wald(m, [n for n in m.params.index if n.startswith("C(klass")])
print(f"LMM class effect chi2({df}) = {chi2:.1f}, p = {p:.3g}")
print("mean of participant medians (ms):", rk.groupby("klass").rt.mean().round(1).to_dict())
print("participants with Internal below both Enter and Exit:",
      int(((rk.pivot(index="subject", columns="klass", values="rt").Internal < rk.pivot(index="subject", columns="klass", values="rt").Enter) &
           (rk.pivot(index="subject", columns="klass", values="rt").Internal < rk.pivot(index="subject", columns="klass", values="rt").Exit)).sum()), "of", rk.subject.nunique())

**Result: reproduced.** Trial class affects RT (chi2(2) = 116.6, p = 4.8e-26); Internal trials are the
fastest (883 ms against 1021 for Enter and 1023 for Exit), in 21 of 24 participants.

## Decoding: one pass over the participants

For each participant (`ofc4.analyse_subject`): a GLM per run with the data set's 16 state columns,
its 'first' and 'err' columns and the ten nuisance time courses; the beta maps z-scored within run
under both readings of O2 ('voxel', 'map'); a linear SVM (cost 1) trained leave-one-run-out on the
16 states and tested on the held-out run's maps, with 10 permuted-label classifiers per fold; the
same for the 8-state space (pairs of state columns summed, which equals modelling the two states as
one event type) and the two-back space (regressors rebuilt from the events), and the 16-state space
with its regressors rebuilt from the events (a check of the rebuilt regressors); and the 16-state
classifier tested on single volumes of the held-out run. Each participant has its own seed (1000 +
participant number). No beta map or volume is smoothed: the files hold no voxel coordinates (O1).

In [ ]:
t = time.time()
R = {s: ofc4.analyse_subject(s, seed=1000 + s) for s in SUBJ}
print(f"decoding done in {time.time() - t:.0f} s")
PRED = pd.concat([r["pred"] for r in R.values()], ignore_index=True)
PERM = pd.concat([r["perm"] for r in R.values()], ignore_index=True)
TRIAL = pd.concat([r["trial"] for r in R.values()], ignore_index=True)
PRED["c"] = PRED.true == PRED.pred
PERM["c"] = PERM.true == PERM.pred
print("test maps per participant, space and reading:", PRED.groupby(["subject", "z", "space"]).size().describe().round(1).to_dict())
miss = {(s, r, k): v for s, res in R.items() for r, d in res["missing_two"].items() for k, v in d.items() if v}
print("two-back states absent from a run (left out of training and test):", miss)
print("rebuilt 16-state regressors against the data set's columns: r from",
      round(min(min(v) for res in R.values() for v in res["rebuilt_r"].values()), 3), "to",
      round(max(max(v) for res in R.values() for v in res["rebuilt_r"].values()), 3))

In [ ]:
# Timing check: the 16-state classifier on single volumes of labelled trials, from the volume at the
# stimulus onset (lag 0) to 5 TRs after it. With the event times on the data's clock, state
# information should rise a few seconds after onset (hemodynamic delay).
# Internal trials repeat the state of the trial before, so the volume at onset still carries the
# state of the previous trial; Enter and Exit trials, whose state differs from the previous trial's,
# give the cleaner contrast.
lagt = TRIAL[TRIAL.vol.str.startswith("lag") & TRIAL.cond.isin(ofc4.STATES)].copy()
lagt["c"] = lagt.true == lagt.pred
lagt["switch"] = lagt.cond.map(ofc4.trial_class) != "Internal"
for subset, sel in (("all labelled trials", lagt), ("Enter and Exit trials", lagt[lagt.switch])):
    lag_acc = sel.groupby(["z", "vol", "subject"]).c.mean().unstack("vol")
    for z in ("voxel", "map"):
        m = lag_acc.loc[z]
        print(f"{subset}, z by {z}: mean 16-way accuracy (%) by lag:", (100 * m.mean()).round(2).to_dict())
        print("   t against 6.25% by lag:", {k: round(float(stats.ttest_1samp(m[k], 1 / 16).statistic), 2) for k in m})
lag_acc = lagt[lagt.switch].groupby(["z", "vol", "subject"]).c.mean().unstack("vol")
fig, ax = plt.subplots(figsize=(5, 3))
for z in ("voxel", "map"):
    m = lag_acc.loc[z]
    ax.errorbar(range(6), 100 * m.mean(), 100 * m.sem(), label=f"z by {z}", capsize=3)
ax.axhline(100 / 16, ls="--", c="k")
ax.set_xlabel("volume after onset (TRs)"); ax.set_ylabel("16-way accuracy (%), Enter and Exit trials"); ax.legend()
print("plotted means range from", round(100 * lag_acc.mean().min(), 2), "to", round(100 * lag_acc.mean().max(), 2), "%")
plt.show()
va = TRIAL[TRIAL.vol.isin(["after", "nearest"]) & TRIAL.cond.isin(ofc4.STATES)].copy()
va["c"] = va.true == va.pred
va["klass"] = va.cond.map(ofc4.trial_class)
print("16-way accuracy (%) on single volumes 2 TRs after onset, labelled trials by class (mean over participants):")
print((100 * va.groupby(["z", "vol", "klass", "subject"]).c.mean().groupby(["z", "vol", "klass"]).mean()).round(2).unstack("klass").to_string())
SHIFT = pd.concat([r["shift"] for r in R.values()], ignore_index=True)
sh = SHIFT.groupby(["subject", "shift"]).acc.mean().unstack()
print("16-way accuracy (%, z by voxel) with the state regressors shifted by k TRs against the time series:")
print("   mean:", (100 * sh.mean()).round(2).to_dict())
print("   t against 6.25%:", {k: round(float(stats.ttest_1samp(sh[k], 1 / 16).statistic), 2) for k in sh})

In [ ]:
# What single volumes of Enter trials are decoded as. An Enter trial follows an Exit trial, and the
# two states differ, so a volume that still carries the trial before is decoded as the state of the
# Exit trial before. Per participant: share of predictions equal to the state of the trial before,
# against the share of all the participant's predictions (labelled trials, same z-scoring and volume)
# that name that state, the rate expected without lag. Error trials of the span reading as well.
tq = TRIAL[TRIAL.vol.isin(["after", "nearest"] + [f"lag{k}" for k in range(6)])].copy()
prev = tq[["z", "vol", "subject", "run", "pos", "cond"]].assign(pos=tq.pos + 1).rename(columns={"cond": "prev_cond"})
tq = tq.merge(prev, on=["z", "vol", "subject", "run", "pos"], how="left")
tq["klass"] = tq.cond.map(lambda x: ofc4.trial_class(x) if x in ofc4.STATES else ("error" if x == "err" else None))
tq = tq[tq.prev_cond.isin(ofc4.STATES)]
base = tq[tq.cond.isin(ofc4.STATES)].groupby(["z", "vol", "subject"]).pred.value_counts(normalize=True).rename("base").reset_index()
tq = tq.merge(base.rename(columns={"pred": "prev_cond"}), on=["z", "vol", "subject", "prev_cond"], how="left").fillna({"base": 0})
tq["as_prev"] = (tq.pred == tq.prev_cond).astype(float)
tq["as_own"] = (tq.pred == tq.true_span).astype(float)
sel = tq[tq.klass.isin(["Enter", "Exit"]) | (tq.klass.eq("error") & tq.true_span.notna())]
lagtab = sel.groupby(["z", "vol", "klass", "subject"])[["as_own", "as_prev", "base"]].mean().groupby(["z", "vol", "klass"])
out = (100 * lagtab.mean()).round(2)
out["t_prev_vs_base"] = sel.groupby(["z", "vol", "klass", "subject"])[["as_prev", "base"]].mean().assign(d=lambda d: d.as_prev - d.base).groupby(["z", "vol", "klass"]).d.apply(lambda d: stats.ttest_1samp(d, 0).statistic).round(2)
out["n_subj"] = lagtab.size()
print("share (%) of single volumes decoded as the trial's own state, as the state of the trial before, and that state's base rate:")
print(out.to_string())
pc = TRIAL[TRIAL.vol.eq("after") & TRIAL.cond.isin(ofc4.STATES)].assign(pred_class=lambda d: d.pred.map(ofc4.trial_class))
print("share (%) of predictions naming a state of each class, labelled trials, volume 'after' (8 Enter, 4 Exit and 4 Internal states: 50, 25 and 25% without bias):")
print((100 * pc.groupby(["z", "subject"]).pred_class.value_counts(normalize=True).unstack().groupby("z").mean()).round(2).to_string())


Timing of the time series against the events. With the state regressors shifted against the time
series, 16-way accuracy is at chance for shifts of -4 and -6 TRs (6.45% and 6.34%), rises to 8.93% at
no shift and stays above chance for shifts of +1 and +2 TRs (8.42% and 9.18%). On single volumes, with
maps z-scored by map, accuracy on labelled trials is at chance at onset (6.08%) and highest 2 to 3 TRs
after it (7.04% and 7.02%). Enter and Exit trials alone, whose state differs from the trial before,
decode near chance at every lag (5.34% to 6.66%), so they cannot show the delay. This is consistent
with the events and the time series sharing a clock to within about two TRs; it does not show it more
precisely. 16-way accuracy also stays above chance with the regressors shifted by +6 TRs (7.79%,
t = 2.18) and -1 TR (7.88%, t = 2.49), so part of the decoded signal changes slowly, over several
trials, as runs of Internal trials that repeat a state would make it.

On the single volume used for the trial-wise claims (2 TRs after onset), accuracy depends strongly
on the trial's class: with maps z-scored by voxel, Enter trials decode at 0.90%, Exit at 11.79% and
Internal at 14.30% (volume 'after'); with maps z-scored by map, 8.30%, 4.85% and 8.04%. Any comparison
of trials of one class with trials of mixed class is therefore confounded by class, which bears on
the error-trial claims (fig4b), where every error trial that can be given a state is an Enter trial.

Why correct Enter trials decode at 0.90% with maps z-scored by voxel, far below chance. If the volume
2 TRs after onset still carried the Exit trial before, Enter trials would be decoded as that Exit
state more often than the classifier names it otherwise. They are not: at the volume 'after' the
share decoded as the previous trial's state is 10.78%, the same as the base rate at which that state
is predicted on all labelled trials (10.90%, t = -0.20); an excess appears only at onset (lag 0,
12.63% against 10.98%, t = 3.12). The low accuracy comes from a bias of the classifier: with maps
z-scored by voxel it names an Enter state on 5.60% of labelled volumes, where the 8 Enter states of
16 would take 50% without bias (z by map, 56.74%). On single volumes of this classifier, Enter trials
are therefore near 0% whatever the signal, and comparisons of Enter error trials with trials of other
classes measure this bias. Error trials of the span reading are decoded as the previous state on
17.38% of volumes against a base rate of 11.93% (t = 1.01, 21 participants), not a significant
excess.


In [ ]:
def comp_acc(df, comps=ofc4.COMPONENTS, index_map=None):
    """Per participant accuracy on each component; index_map gives the label position of each component."""
    index_map = index_map or {"cur_cat": 0, "cur_age": 1, "prev_cat": 2, "prev_age": 3}
    out = {}
    for comp in comps:
        i = index_map[comp]
        out[comp] = (df.true.str[i] == df.pred.str[i]).groupby(df.subject).mean()
    return pd.DataFrame(out)


def ttest(x, mu, alternative="greater"):
    r = stats.ttest_1samp(x, mu, alternative=alternative)
    return float(r.statistic), float(r.pvalue)


def boot_p(x, mu, n=10000, seed=0):
    """Bootstrap over participants (Supplemental Results 1.2): share of resampled means at or below mu."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    means = x[rng.integers(0, len(x), (n, len(x)))].mean(1)
    return float((means <= mu).mean())


ACC = PRED.groupby(["z", "space", "subject"]).c.mean()

## fig2a-16way-decoding

> The 16 task states can be decoded from multivoxel patterns in anatomically defined OFC above chance (mean 12.2% against 6.25%).

Readings: z-scoring O2 ('voxel', 'map'); the t-test side O4 (one- and two-sided). Smoothing (O1)
can only be run unsmoothed. Post-error trials (O3): the data set's 'err' column follows a short event
at each block onset (data check), so with the data set's columns ('16') the trials inside a block after
the error trial are in no regressor, which is neither reading of O3. Both readings are rebuilt from
the 'err' events (`ofc4.subject_designs`): (a) '16o3a', the 'err' column with a separate nuisance
boxcar from onset + 4.5 s (the mean trial duration, Methods, Task) to the end of the block; (b)
'16o3b', one 'err' boxcar over the whole block.

In [ ]:
for z in ("voxel", "map"):
    for space in ("16", "16rebuilt", "16o3a", "16o3b"):
        a = ACC.loc[(z, space)]
        t1, p1 = ttest(a, 1 / 16)
        t2, p2 = ttest(a, 1 / 16, "two-sided")
        print(f"z by {z}, {space}: mean {100 * a.mean():.2f}% (SEM {100 * a.sem():.2f}), above chance in {int((a > 1 / 16).sum())} of {len(a)}; "
              f"t{len(a) - 1} = {t1:.2f}, one-sided p = {p1:.2g}, two-sided p = {p2:.2g}")
print("spaces: 16 = the data set's columns; 16rebuilt = state regressors rebuilt from the events; 16o3a and 16o3b = the data set's\n"
      "state columns with the two readings of O3 for the error blocks (a: separate post-error regressor, b: 'err' over the whole block)")
print("expected 12.2 +/- 1.22; per participant (%):")
print((100 * ACC.loc[(slice(None), "16")].unstack("z")).round(2).T.to_string())

**Result: partially reproduced.** 16-way decoding is above chance under both z-scorings (8.93% and
8.92%, one-sided p = 0.0021 and 0.00037, two-sided p = 0.0041 and 0.00073), but below the 10.98 to
13.42% of the tolerance. With regressors rebuilt from the events it is 9.57% and 9.19%. Under the two
readings of O3 it is 8.97% and 9.38% (z by voxel) and 8.37% and 8.10% (z by map), all above chance
(one-sided p <= 0.0045) and all below the tolerance, so the outcome is the same under every reading. The maps are
unsmoothed, as the files hold no coordinates; the paper smoothed them.

## fig2-two-back-state-space

> A classifier for an alternative 16-state space built from the task-irrelevant category and age of
> two trials back decodes worse than the hypothesized state space (8.1% against 12.2%).

Each labelled trial is relabelled by its current category and age and the category and age of the
trial two back (the previous components of the trial before it). Readings for a trial whose trial
before is the first trial or an error block: (a) left out, as one nuisance regressor; (b) after a
block with one reconstructed error, which is an Enter error repeated as the same trial (Methods,
Task), the trial two back as presented is the error trial, whose state is that of its repetition,
the trial before; others left out. The hypothesized space is compared both as the data set's columns
('16') and as regressors rebuilt from the events the same way as the two-back ones ('16rebuilt').

In [ ]:
for z in ("voxel", "map"):
    for hyp in ("16", "16rebuilt"):
        h = ACC.loc[(z, hyp)]
        for reading in ("a", "b"):
            a = ACC.loc[(z, "two" + reading)]
            t, p = stats.ttest_rel(a, h)
            print(f"z by {z}, against {hyp}, reading {reading}: two-back {100 * a.mean():.2f}% against hypothesized {100 * h.mean():.2f}%; "
                  f"paired t{len(a) - 1} = {t:.2f}, two-sided p = {p:.3g}; two-back lower in {int((a < h).sum())} of {len(a)}")
print("expected two-back 8.1 +/- 0.81")
print("labelled trials left out of the two-back states (nuisance regressor):",
      {k: sum(res["two_leftout"][r][k] for res in R.values() for r in res["two_leftout"]) for k in "ab"},
      "of", int(EV.is_state.sum()))

**Result: partially reproduced.** The two-back space decodes worse than the hypothesized one, but no
reading meets both parts of the criterion. With maps z-scored by voxel (the assumption), it is lower
(5.96% against 8.93%, t23 = -3.75, p = 0.001; reading b 6.17%, p = 0.003; against the rebuilt
hypothesized space 9.57%, p = 6e-5 and 0.0003), but its accuracy is at chance, below the 7.29 to 8.91%
of the tolerance. With maps z-scored by map, its accuracy is within tolerance (7.88% in both readings)
but not significantly lower (p = 0.13 and 0.14 against '16', 0.070 and 0.073 against '16rebuilt').

## fig2b-hidden-components and fig2b-current-age-absent

> Within the 16-way classification, the three hidden, task-relevant components (previous category,
> previous age, current category) are each classified above chance.

> The observable component, current age, is not classified above chance within the 16-way classification.

Readings: one-sided t-test against 50%, and the bootstrap of participants (10000 resamples,
Supplemental Results 1.2), whose p is the share of resampled means at or below 50%.

In [ ]:
COMP16 = {}
for z, space in [(z, sp) for sp in ("16", "16o3a", "16o3b") for z in ("voxel", "map")]:
    ca = comp_acc(PRED[(PRED.z == z) & (PRED.space == space)])
    if space == "16":
        COMP16[z] = ca
    for comp in ["prev_cat", "prev_age", "cur_cat", "cur_age"]:
        t, p = ttest(ca[comp], 0.5)
        print(f"{space}, z by {z}, {comp}: mean {100 * ca[comp].mean():.2f}% (SEM {100 * ca[comp].sem():.2f}), t{len(ca) - 1} = {t:.2f}, "
              f"one-sided p = {p:.3g}, bootstrap p = {boot_p(ca[comp], 0.5, seed=11):.4f}, above 50% in {int((ca[comp] > 0.5).sum())} of {len(ca)}")
print("paper: previous category p = 0.014, previous age p = 0.003, current category p = 0.007 (t26 > 3); current age p > 0.72")

**Result: partially reproduced.** Of the three hidden components, only the previous category is above
50% (52.03%, one-sided p = 0.048; bootstrap p = 0.046; z by map 51.78%, p = 0.038). Previous age (49.08%,
p = 0.77) and current category (50.26%, p = 0.42) are not; z-scored by map they are 51.16% (p = 0.16)
and 51.78% (p = 0.10). Under the two readings of O3 the pattern is the same: previous category
reaches p < 0.05 in three of four (p = 0.013 to 0.034; 0.12 for O3 a, z by voxel), more strongly than
in the main reading for O3 (a) with maps z-scored by map (p = 0.013 against 0.038), previous age and
current category in none (p >= 0.10).

**Result: reproduced.** Current age is not above chance (50.83%, one-sided p = 0.27; z by map 47.92%,
p = 0.95; bootstrap p = 0.27 and 0.97), nor under the two readings of O3 (p = 0.094 to 0.86).

## fig2b-two-back-absent

> The task-irrelevant category and age from two trials back are not classified above chance by the two-back classifier.

In [ ]:
for z in ("voxel", "map"):
    for reading in ("a", "b"):
        ca = comp_acc(PRED[(PRED.z == z) & (PRED.space == "two" + reading)])
        for comp, name in (("prev_cat", "two-back category"), ("prev_age", "two-back age")):
            t, p = ttest(ca[comp], 0.5)
            print(f"z by {z}, reading {reading}, {name}: mean {100 * ca[comp].mean():.2f}%, t{len(ca) - 1} = {t:.2f}, one-sided p = {p:.3g}, "
                  f"bootstrap p = {boot_p(ca[comp], 0.5, seed=12):.4f}")
        print(f"   current category {100 * ca['cur_cat'].mean():.2f}%, current age {100 * ca['cur_age'].mean():.2f}%")

**Result: reproduced.** Neither two-back component is above chance in any reading by the t-test
(smallest one-sided p = 0.055, the two-back category with maps z-scored by map, reading a). The
bootstrap variant gives p = 0.042 for that component (0.081 in reading b), and p > 0.5 with maps
z-scored by voxel.

## figs2c-8way-decoding

> Decoding an 8-state space without current age is above chance (16.7%), and its three components are
> decoded above baseline (53.5%, 52.9%, 55.5%).

t-tests against 12.5% and 50%, one- and two-sided (the side is not stated; O4).

In [ ]:
for z in ("voxel", "map"):
    a = ACC.loc[(z, "8")]
    t1, p1 = ttest(a, 1 / 8)
    t2, p2 = ttest(a, 1 / 8, "two-sided")
    print(f"z by {z}: 8-way mean {100 * a.mean():.2f}% (SEM {100 * a.sem():.2f}), t{len(a) - 1} = {t1:.2f}, one-sided p = {p1:.3g}, two-sided p = {p2:.3g}")
    ca = comp_acc(PRED[(PRED.z == z) & (PRED.space == "8")], ["prev_cat", "prev_age", "cur_cat"],
                  {"cur_cat": 0, "prev_cat": 1, "prev_age": 2})
    for comp in ca:
        t1, p1 = ttest(ca[comp], 0.5)
        t2, p2 = ttest(ca[comp], 0.5, "two-sided")
        print(f"   {comp}: {100 * ca[comp].mean():.2f}%, t = {t1:.2f}, one-sided p = {p1:.3g}, two-sided p = {p2:.3g}")
print("expected: 8-way 16.7 +/- 1.67; previous category 53.5, previous age 52.9, current category 55.5 (+/- 10%)")

**Result: reproduced, resting on the voxel z-scoring.** With maps z-scored by voxel, 8-way accuracy is
15.89% (within 15.03 to 18.37%), above 12.5% (one-sided p = 0.006, two-sided p = 0.012), and previous
category (55.36%, p = 0.0024) and current category (54.48%, p = 0.0036) are above 50%. Previous age is
not (49.09%, p = 0.67), which fits neither the main text ("above baseline") nor the supplement's
marginal p = .09. With maps z-scored by map, accuracy is 14.79% (outside tolerance, one-sided p = 0.025)
and no component is above 50% (p = 0.20 to 0.44).

## figs2a-permutation-control

> Classifiers trained on permuted labels decode at chance, and the real decoding is above the
> permutation results for the 16-way classification and each binary component.

10 permutations of the training labels per fold and participant, test labels unchanged, accuracies
averaged within participant (Supplemental Results 1.2). Readings: (a) 95th percentile over the
participants' averaged permutation accuracies, paired t-test of real against permuted; (b) 95th
percentile over all participant and permutation values, compared with the real group mean.

In [ ]:
for z in ("voxel", "map"):
    q = PERM[PERM.z == z]
    real16 = ACC.loc[(z, "16")]
    perm_sub = q.groupby(["subject", "perm"]).c.mean()
    pa = perm_sub.groupby("subject").mean()
    t, p = ttest(pa, 1 / 16)
    print(f"z by {z}: permuted 16-way mean {100 * pa.mean():.2f}%, t against 6.25% = {t:.2f}, one-sided p = {p:.3f}")
    rows = [("16-way", real16, pa, perm_sub)]
    for comp in ["prev_cat", "prev_age", "cur_cat", "cur_age"]:
        i = {"cur_cat": 0, "cur_age": 1, "prev_cat": 2, "prev_age": 3}[comp]
        cc = (q.true.str[i] == q.pred.str[i]).groupby([q.subject, q.perm]).mean()
        rows.append((comp, COMP16[z][comp], cc.groupby("subject").mean(), cc))
    for name, real, permavg, pooled in rows:
        tt, pp = stats.ttest_rel(real, permavg.reindex(real.index), alternative="greater")
        print(f"   {name}: real {100 * real.mean():.2f}%, permuted {100 * permavg.mean():.2f}%; (a) 95th pct {100 * np.percentile(permavg, 95):.2f}%, "
              f"paired t = {tt:.2f}, one-sided p = {pp:.3g}; (b) 95th pct {100 * np.percentile(pooled, 95):.2f}%, real mean above it: {real.mean() > np.percentile(pooled, 95)}")
print("expected 95th percentiles: 16-way 7.04, previous category 51.5, previous age 51.7, current category 51.4, current age 51.7 (+/- 10%)")

**Result: partially reproduced.** Permuted classifiers decode at chance (5.98% and 5.94%, not above
6.25%), and real 16-way accuracy exceeds the permuted one (paired one-sided p = 0.001 and 0.0003). No
component meets p < 0.025: previous category comes closest (p = 0.033 and 0.031), then current category with maps z-scored by
map (p = 0.049). The 95th percentiles
of the participants' averaged permutation accuracies (reading a) are 7.09% for the 16-way and 51.69,
52.09, 51.84 and 52.00% for the components, all within tolerance of the paper's 7.04, 51.5, 51.7, 51.4
and 51.7%; over pooled participant and permutation values (reading b) they are 10.0% and about 57.5%.

## Pattern similarity (fig2c-enter-cluster, fig5-similarity-predicts-errors, figs5-euclidean-distance)

Methods, State Space Similarities: each run's state maps against all maps of the other runs, the
matrices averaged over runs, negative correlations set to NaN. Readings for the maps (O1, O2): raw
betas, or z-scored within run by voxel or by map as for classification.

In [ ]:
ENTER = np.array([ofc4.trial_class(s) == "Enter" for s in ofc4.STATES])


def between_run(betas, rs, prep, metric):
    """16 x 16 between-run similarity (Pearson r) or distance (Euclidean), averaged over ordered run pairs."""
    maps = {r: betas[("16", r)] if prep == "raw" else ofc4.zscore_maps(betas[("16", r)], prep) for r in rs}
    mats = []
    for r in rs:
        for r2 in rs:
            if r2 == r:
                continue
            A, B = maps[r], maps[r2]
            if metric == "corr":
                Az = (A - A.mean(1, keepdims=True)) / A.std(1, keepdims=True)
                Bz = (B - B.mean(1, keepdims=True)) / B.std(1, keepdims=True)
                mats.append(Az @ Bz.T / A.shape[1])
            else:
                mats.append(np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)))
    M = np.mean(mats, axis=0)
    if metric == "corr":
        M[M < 0] = np.nan
    return M


SIM = {(prep, metric, s): between_run(R[s]["betas"], sorted({k[1] for k in R[s]["betas"]}), prep, metric)
       for prep in ("raw", "voxel", "map") for metric in ("corr", "dist") for s in SUBJ}
for prep in ("raw", "voxel", "map"):
    allc = np.array([SIM[(prep, "corr", s)] for s in SUBJ])
    print(f"{prep}: share of NaN (negative) correlations {np.isnan(allc).mean():.3f}; correlations from {np.nanmin(allc):.3f} to {np.nanmax(allc):.3f}")

## fig2c-enter-cluster

> In the similarity structure of OFC state patterns, pairs of states are more similar when both are
> Enter or both non-Enter states than when they differ.

Per participant the mean within-class (Enter with Enter, non-Enter with non-Enter) against the mean
between-class correlation, paired t-test. Readings: the diagonal left out or kept; maps raw or z-scored.

In [ ]:
same = ENTER[:, None] == ENTER[None, :]
for prep in ("raw", "voxel", "map"):
    for diag in (False, True):
        w, b = [], []
        for s in SUBJ:
            M = SIM[(prep, "corr", s)]
            mask_w = same.copy()
            if not diag:
                np.fill_diagonal(mask_w, False)
            w.append(np.nanmean(M[mask_w]))
            b.append(np.nanmean(M[~same]))
        w, b = np.array(w), np.array(b)
        t, p = stats.ttest_rel(w, b)
        print(f"{prep}, diagonal {'in' if diag else 'out'}: within {w.mean():.4f}, between {b.mean():.4f}, paired t{len(w) - 1} = {t:.2f}, "
              f"two-sided p = {p:.3g}; within > between in {int((w > b).sum())} of {len(w)}")
# dendrogram of the group matrix (shown only; the linkage method is not given, average linkage is a standard choice)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
G = np.nanmean([SIM[("voxel", "corr", s)] for s in SUBJ], axis=0)
G = (G + G.T) / 2
D = 1 - np.nan_to_num(G, nan=0.0)
np.fill_diagonal(D, 0)
fig, ax = plt.subplots(figsize=(7, 3))
labels_paper = [f"({s[2:]}){s[:2]}" + ("*" if e else "") for s, e in zip(ofc4.STATES, ENTER)]
dendrogram(linkage(squareform(D, checks=False), "average"), labels=labels_paper, ax=ax)
ax.set_ylabel("1 - r (z by voxel)"); ax.set_title("* Enter states")
print("group 1 - r ranges from", round(D[~np.eye(16, dtype=bool)].min(), 3), "to", round(D.max(), 3))
plt.show()

**Result: reproduced.** Within-class correlations exceed between-class ones (raw or map-z-scored maps,
diagonal left out: 0.0355 against 0.0320, t23 = 2.82, p = 0.0097, 20 of 24 participants; diagonal in:
p = 0.0019). With maps z-scored by voxel, p = 0.052 without the diagonal and 0.0023 with it.
Correlations are small (at most 0.257) and 20% of them (50% after voxel z-scoring) are negative and set
to NaN.

## fig2d-enter-nonenter-decoding

> All three hidden components are decoded above chance even when the 16-way classifier is tested
> separately on Enter and on non-Enter trials.

The predictions of the 16-way classifiers, split by the class of the test map; component accuracy in
each subset. Readings: (a) per component the mean of the Enter and non-Enter accuracies; (b) each of
the six. One-sided t-tests against 50% (two-sided p shown).

In [ ]:
for z, space in [(z, sp) for sp in ("16", "16o3a", "16o3b") for z in ("voxel", "map")]:
    q = PRED[(PRED.z == z) & (PRED.space == space)]
    q = q.assign(enter=q.true.map(lambda s: ofc4.trial_class(s) == "Enter"))
    sub = {k: comp_acc(q[q.enter == v], ["prev_cat", "prev_age", "cur_cat"]) for k, v in (("Enter", True), ("non-Enter", False))}
    avg = (sub["Enter"] + sub["non-Enter"]) / 2
    for comp in avg:
        t, p = ttest(avg[comp], 0.5)
        print(f"{space}, z by {z}, (a) {comp}: {100 * avg[comp].mean():.2f}%, t = {t:.2f}, one-sided p = {p:.3g}, two-sided p = {min(1, 2 * min(p, 1 - p)):.3g}")
    for k, ca in sub.items():
        for comp in ca:
            t, p = ttest(ca[comp], 0.5)
            print(f"   (b) {k} {comp}: {100 * ca[comp].mean():.2f}%, t = {t:.2f}, one-sided p = {p:.3g}")

**Result: not reproduced.** Tested separately on Enter and non-Enter maps, only the previous category
reaches p < 0.05 in reading (a) (52.03%, one-sided p = 0.048; z by map 51.78%, p = 0.038); previous age
and current category do not (p >= 0.10). In reading (b) at most one of six reaches p < 0.05. Under the
two readings of O3 the same holds: in reading (a) only previous category reaches p < 0.05 (three of
four), and in reading (b) at most one of six.

## fig4a-decoding-error-correlation

> Across participants, mean classification accuracy of the three hidden components in OFC correlates
> negatively with error rate.

The mean of the three hidden-component accuracies (fig2b-hidden-components) against the error rate
(behavior-error-rate, reading b, the error blocks over the 96 trials after the first of each run);
Pearson correlation and bootstrap 95% CI over participants (percentile intervals, 10000 and 1000 resamples).

In [ ]:
for z in ("voxel", "map"):
    hid = COMP16[z][["prev_cat", "prev_age", "cur_cat"]].mean(axis=1)
    for name, err in (("reading b", ERRRATE), ("reading a", er.rate_a / 100)):
        x, y = hid.reindex(SUBJ).values, err.reindex(SUBJ).values
        r, p = stats.pearsonr(x, y)
        line = f"z by {z}, error rate {name}: r = {r:.3f}, t{len(x) - 2} = {r * np.sqrt((len(x) - 2) / (1 - r ** 2)):.2f}, p = {p:.3g}"
        for nb, seed in ((10000, 21), (1000, 22)):
            rng = np.random.default_rng(seed)
            bs = []
            for _ in range(nb):
                i = rng.integers(0, len(x), len(x))
                if np.std(x[i]) > 0 and np.std(y[i]) > 0:
                    bs.append(np.corrcoef(x[i], y[i])[0, 1])
            lo, hi = np.percentile(bs, [2.5, 97.5])
            line += f"; {nb} resamples CI [{lo:.3f}, {hi:.3f}]"
        print(line)
    sp = stats.spearmanr(hid.reindex(SUBJ), ERRRATE.reindex(SUBJ))
    print(f"   Spearman rho = {sp.statistic:.3f}, p = {sp.pvalue:.3g} (variant; participant 9 has the highest error rate, {100 * ERRRATE.max():.2f}%)")
print("expected r = -0.58 +/- 0.2632 (paper CI -0.8432 to -0.0169)")
fig, ax = plt.subplots(figsize=(4, 3))
hid = COMP16["voxel"][["prev_cat", "prev_age", "cur_cat"]].mean(axis=1)
ax.scatter(100 * ERRRATE.reindex(SUBJ), 100 * hid.reindex(SUBJ))
ax.set_xlabel("error rate (%)"); ax.set_ylabel("mean hidden-component accuracy (%), z by voxel")
print("plotted ranges: error rate", round(100 * ERRRATE.min(), 2), "to", round(100 * ERRRATE.max(), 2),
      "; accuracy", round(100 * hid.min(), 2), "to", round(100 * hid.max(), 2))
plt.show()

**Result: partially reproduced, resting on the voxel z-scoring.** The correlation is negative but weak
and not significant (r = -0.268, p = 0.21, bootstrap CI -0.717 to 0.338); it lies inside the paper's CI
(-0.843 to -0.017) but outside the tolerance (-0.843 to -0.317). With maps z-scored by map, r = -0.004,
outside the paper's CI. The error rate counts one error per block, a lower bound.

## Trial-wise decoding around errors (fig4b-error-trial-drop, fig4b-decline-before-errors, fig4b-error-vs-matched)

The 16-way classifier of each fold tested on single volumes of the held-out run, z-scored per voxel
over the run (Methods), unsmoothed (no voxel coordinates). Readings: the volume (first at or after
onset + 4.8 s, or nearest), the z-scoring of the training maps, and the error trial's state (strict:
runs where every block holds one designed trial; span: Enter errors whose block spans two trials).
An error enters when its five preceding trials are labelled trials of the same run (Methods,
Participants); trial +1 is the first labelled trial after the block. Matched courses: (a) every
non-overlapping run of seven consecutive labelled trials, (b) one such run per error, drawn from the
error's run outside its course.

In [ ]:
def courses(z, vol, err_reading, seed):
    rng = np.random.default_rng(seed)
    q = TRIAL[(TRIAL.z == z) & (TRIAL.vol == vol)].copy()
    q["truth"] = q["true"] if err_reading == "strict" else q["true_span"]
    q["c"] = (q.truth == q.pred).astype(float)
    q["cls"] = q.truth.map(lambda x: ofc4.trial_class(x) if isinstance(x, str) else None)
    err_rows, mat_a, mat_b = [], [], []
    for (s, r), g in q.groupby(["subject", "run"]):
        g = g.set_index("pos").sort_index()
        lab_pos = set(g.index[g.cond.isin(ofc4.STATES)])
        used = set()
        for p, e in g[g.cond == "err"].iterrows():
            if pd.isna(e.truth) or not all((p - k) in lab_pos for k in range(1, 6)):
                continue
            eid = f"{s}-{r}-{p}"
            for k in range(-5, 2):
                pp = p + k
                if k == 0:
                    err_rows.append({"subject": s, "eid": eid, "rel": 0, "c": e.c, "cls": e.cls})
                elif pp in lab_pos:
                    err_rows.append({"subject": s, "eid": eid, "rel": k, "c": g.loc[pp, "c"], "cls": g.loc[pp, "cls"]})
            course = set(range(p - 5, p + 2))
            starts = [st for st in sorted(lab_pos) if all((st + k) in lab_pos for k in range(7)) and not (set(range(st, st + 7)) & course)]
            if starts:
                st = starts[rng.integers(len(starts))]
                for k in range(7):
                    mat_b.append({"subject": s, "eid": eid, "rel": k - 5, "c": g.loc[st + k, "c"]})
        st = min(lab_pos) if lab_pos else None
        positions = sorted(lab_pos)
        i = 0
        while i < len(positions):
            st = positions[i]
            if all((st + k) in lab_pos for k in range(7)):
                for k in range(7):
                    mat_a.append({"subject": s, "eid": f"{s}-{r}-m{st}", "rel": k - 5, "c": g.loc[st + k, "c"], "cls": g.loc[st + k, "cls"]})
                i += 7
            else:
                i += 1
    return pd.DataFrame(err_rows), pd.DataFrame(mat_a), pd.DataFrame(mat_b)


COURSES = {(z, vol, er_): courses(z, vol, er_, seed=31) for z in ("voxel", "map") for vol in ("after", "nearest") for er_ in ("strict", "span")}
for key, (e, ma, mb) in COURSES.items():
    print(key, f"errors {e.eid.nunique()} from {e.subject.nunique()} participants, by class {e[e.rel == 0].cls.value_counts().to_dict()}; "
          f"matched (a) {ma.eid.nunique()} courses, (b) {mb.eid.nunique()}; (a) with an Enter trial at position 0 {ma[(ma.rel == 0) & (ma.cls == 'Enter')].eid.nunique()}")

## fig4b-error-trial-drop

> Trial-wise 16-way decoding is lower on error trials (4.2%) than in the five trials preceding an error (11%).

In [ ]:
for key, (e, ma, mb) in COURSES.items():
    pre = e[e.rel.between(-5, -1)].groupby("subject").c.mean()
    at = e[e.rel == 0].groupby("subject").c.mean()
    t, p = stats.ttest_rel(pre, at.reindex(pre.index))
    print(f"{key}: n = {len(pre)} participants, {e.eid.nunique()} errors; five preceding {100 * pre.mean():.2f}%, error trials {100 * at.mean():.2f}%; "
          f"paired t{len(pre) - 1} = {t:.2f}, two-sided p = {p:.3g}")
print("expected: preceding 11 +/- 1.1, error 4.2 +/- 0.42")
lab_trials = TRIAL[(TRIAL.vol == "after") & TRIAL.cond.isin(ofc4.STATES)]
print("all labelled trials, volume 'after':", (100 * (lab_trials.true == lab_trials.pred).groupby(lab_trials.z).mean()).round(2).to_dict())

**Result: not reproduced.** Error trials decode worse than the five trials before them with maps
z-scored by voxel (1.59% against 8.02%, 21 participants, 81 errors), but not significantly
(p = 0.061; 0.18 with the 37 strictly reconstructed errors of 16 participants), and both values are
outside tolerance (expected 4.2% and 11%). With maps z-scored by map, error trials decode better than
the preceding ones (7.70% against 5.67%). Only error trials whose state can be reconstructed enter, and
all are Enter trials, which decode at 0.90% (z by voxel) on correct trials too, as that classifier rarely names an
Enter state (data check), so the
comparison with the mixed-class trials before them is confounded by class.

## fig4b-decline-before-errors

> Decoding accuracy declines linearly over the trials leading up to and including an error, and no
> such decline is seen in matched time courses without errors.

Linear mixed models of accuracy on trial position (-5 to 0), random intercept and slope by
participant, Wald chi-square; accuracy entered per trial (0 or 1) or as the mean per participant and
position; the interaction from a joint model of error and matched courses.

In [ ]:
def slope_test(df, formula, term, re="~rel"):
    """Random intercept and slope; when that fit does not converge, the random-intercept model
    is used and the result is marked."""
    try:
        m = smf.mixedlm(formula, df, groups=df["subject"], re_formula=re).fit()
        slope_ok = bool(m.converged)
    except np.linalg.LinAlgError:  # singular fit: counted as not converged
        slope_ok = False
    if not slope_ok:
        try:
            m = smf.mixedlm(formula, df, groups=df["subject"]).fit()
        except np.linalg.LinAlgError:
            print("   a random-intercept fit was singular; its result is left out:", formula, f"({len(df)} rows)")
            return np.nan, np.nan, np.nan, False
    chi2, _, p = wald(m, [term])
    return m.params[term], chi2, p, slope_ok


DECL = {}
for key, (e, ma, mb) in COURSES.items():
    for mname, mdf in (("a", ma), ("b", mb)):
        for unit in ("trial", "mean"):
            ee = e[e.rel <= 0].assign(cond="error")
            mm = mdf[mdf.rel <= 0].assign(cond="matched")
            if unit == "mean":
                ee = ee.groupby(["subject", "rel", "cond"]).c.mean().reset_index()
                mm = mm.groupby(["subject", "rel", "cond"]).c.mean().reset_index()
            b1, c1, p1, v1 = slope_test(ee, "c ~ rel", "rel")
            b2, c2, p2, v2 = slope_test(mm, "c ~ rel", "rel")
            j = pd.concat([ee, mm])
            j["is_err"] = (j.cond == "error").astype(int)
            b3, c3, p3, v3 = slope_test(j, "c ~ rel * is_err", "rel:is_err")
            DECL[(key, mname, unit)] = (b1, p1, b3, p3, b2, p2)
            print(f"{key}, matched {mname}, {unit}{'' if (v1 and v2 and v3) else ' (random intercept only in a model)'}: error slope {100 * b1:.2f} pts/trial chi2 = {c1:.2f} p = {p1:.3g} | "
                  f"interaction {100 * b3:.2f} chi2 = {c3:.2f} p = {p3:.3g} | matched slope {100 * b2:.2f} chi2 = {c2:.2f} p = {p2:.3g}")
print("paper: error slope chi2(1) = 4.7, p = 0.03; interaction chi2(1) = 4.3, p = 0.04; non-error slope p = 0.33")
e, ma, mb = COURSES[("voxel", "after", "span")]
fig, ax = plt.subplots(figsize=(5, 3))
for lbl, df in (("error courses", e), ("matched (a)", ma)):
    g = df.groupby(["subject", "rel"]).c.mean().unstack()
    ax.errorbar(g.columns, 100 * g.mean(), 100 * g.sem(), label=lbl, capsize=3, marker="o")
    print(lbl, "mean accuracy (%) by position:", (100 * g.mean()).round(2).to_dict())
ax.axhline(100 / 16, ls="--", c="k"); ax.set_xlabel("trial relative to error"); ax.set_ylabel("16-way accuracy (%)"); ax.legend()
plt.show()

**Result: not reproduced.** In no reading is the slope over trials -5 to 0 negative with p < 0.05
together with a significant interaction. With maps z-scored by voxel, the error slope is negative but
not significant (smallest p = 0.32) and the interaction is not significant. With maps z-scored by map
and the nearest volume, the slope and interaction are significant but positive (accuracy rises towards
the error; span reading, matched courses (a), slope 1.47 percentage points per trial, p = 0.039;
interaction p = 0.030; strict reading, slope 2.08, p = 0.014). Most random-slope fits did not converge, and the random-intercept models are
reported in their place, as marked; one singular fit is left out. Position 0 is always an Enter trial
and position -1 an Exit trial, so the course also reflects the decoding of each class (data check). See the printed table for every reading.

## fig4b-error-vs-matched

> Decoding on the error trial is lower than on matched correct trials, and marginally lower on the trial before the error.

In [ ]:
for key, (e, ma, mb) in COURSES.items():
    for mname, mdf in (("a", ma), ("b", mb)):
        out = []
        for rel in (0, -1):
            x = e[e.rel == rel].groupby("subject").c.mean()
            y = mdf[mdf.rel == rel].groupby("subject").c.mean().reindex(x.index)
            ok = y.notna()
            t, p1 = stats.ttest_rel(x[ok], y[ok], alternative="less")
            _, p2 = stats.ttest_rel(x[ok], y[ok])
            out.append(f"trial {rel}: error {100 * x[ok].mean():.2f}% vs matched {100 * y[ok].mean():.2f}%, t{int(ok.sum()) - 1} = {t:.2f}, one-sided p = {p1:.3g}, two-sided p = {p2:.3g}")
        print(key, f"matched {mname}:", " | ".join(out))
# Reading (c): every error trial here is an Enter trial, so position 0 is compared with matched courses (a)
# whose position 0 is an Enter trial, and with all labelled Enter trials of the participant.
for key, (e, ma, mb) in COURSES.items():
    x = e[e.rel == 0].groupby("subject").c.mean()
    y = ma[(ma.rel == 0) & (ma.cls == "Enter")].groupby("subject").c.mean().reindex(x.index)
    ok = y.notna()
    t, p1 = stats.ttest_rel(x[ok], y[ok], alternative="less")
    q = TRIAL[(TRIAL.z == key[0]) & (TRIAL.vol == key[1]) & TRIAL.cond.isin(ofc4.STATES)]
    en = (q.true == q.pred)[q.cond.map(ofc4.trial_class) == "Enter"].groupby(q.subject).mean().reindex(x.index)
    t2, p2 = stats.ttest_rel(x, en, alternative="less")
    print(key, f"(c) trial 0: error {100 * x[ok].mean():.2f}% vs matched with Enter at 0 {100 * y[ok].mean():.2f}%, t{int(ok.sum()) - 1} = {t:.2f}, one-sided p = {p1:.3g} | "
          f"error {100 * x.mean():.2f}% vs all labelled Enter trials {100 * en.mean():.2f}%, t{len(x) - 1} = {t2:.2f}, one-sided p = {p2:.3g}")
print("paper: trial 0 t24 = 2.3, p = 0.01; trial -1 t23 = 1.5, p = 0.07")

**Result: not reproduced, resting on the class-matched reading (c).** Every error trial that can be
given a state is an Enter trial, where the paper's errors were of all three classes, and correct Enter
trials decode far worse than others (0.90% against 11.79% and 14.30%, z by voxel), as that
classifier names an Enter state on 5.60% of volumes (data check). Matched to their
class, error trials are not lower: 1.59% against 1.42% at position 0 of matched courses (a) with an
Enter trial there (t20 = 0.18, one-sided p = 0.57) and against 0.88% on all labelled Enter trials
(p = 0.68), and p >= 0.45 in every reading. The readings of the criterion, which do not match class,
give different outcomes: with maps z-scored by voxel and matched courses (a), error trials are lower
(1.59% against 9.21%, t20 = -4.07, one-sided p = 0.0003; strict 2.08% against 8.93%, p = 0.005); with
one matched course per error (b), p = 0.044 to 0.14; with maps z-scored by map the error trial is not
lower (p >= 0.69). Reading (c) is adopted because it compares like with like. That the paper's matched
courses had the class mix of its error trials is an inference: the Figure 4B legend says only
"seven consecutive trials with no behavioral error". With maps z-scored by voxel, reading (c) is at
floor (1.59% against 1.42% and 0.88%, below the 6.25% chance level) and could not show a drop. The
reading that could is the map z-scoring, where labelled Enter trials decode at 8.02%: there the error
trial is not lower under the criterion's readings (p >= 0.69) or under (c) (p >= 0.45). Trial -1 is not lower with maps z-scored by voxel (11.59% against
8.81%, one-sided p = 0.70; paper p = 0.07). The degrees of freedom are 20 (21 participants), against
t24 in the paper.

## fig5-similarity-predicts-errors and figs5-euclidean-distance

> Across the eight Exit to Enter transitions, transitions between states whose OFC patterns are more
> correlated have lower error rates.

> The relation between pattern similarity and error rate holds with between-run Euclidean distances
> in place of correlations.

For each of the eight Enter states, the Exit state before it (its previous category and age, after an
age change) and the error rate on the Enter trial: errors whose state is reconstructed (span or strict
reading) over those errors plus the correct trials of that state. Enter errors whose current age is
unknown cannot be given a transition and are left out. Readings: (a) regression over the eight
transition types of the group-mean error rate on the group-mean similarity; (b) over the eight
ordinal positions of Figure 5B (NaN correlations, the negative ones, ranked lowest); Figure 5C, each
participant's Pearson correlation over the transitions, t-test against 0; and a mixed model of error
rate on similarity, random intercept and slope by participant.

In [ ]:
ENTER_STATES = [s for s in ofc4.STATES if ofc4.trial_class(s) == "Enter"]
EXIT_OF = {s: s[2:] + s[2] + ("y" if s[3] == "o" else "o") for s in ENTER_STATES}
assert all(ofc4.trial_class(x) == "Exit" for x in EXIT_OF.values())


def transition_table(err_reading, prep, metric):
    col = "state_span" if err_reading == "span" else "state"
    rows = []
    for s in SUBJ:
        M = SIM[(prep, metric, s)]
        M = (M + M.T) / 2
        for en in ENTER_STATES:
            ex = EXIT_OF[en]
            n_ok = int(((lab.subject == s) & (lab.cond == en)).sum())
            n_err = int(((ERR.subject == s) & (ERR[col] == en)).sum())
            rows.append({"subject": s, "enter": en, "exit": ex, "sim": M[ofc4.STATES.index(ex), ofc4.STATES.index(en)],
                         "rate": n_err / (n_ok + n_err), "n_ok": n_ok})
    return pd.DataFrame(rows)


def fig5_tests(tt, metric):
    g = tt.groupby("enter")[["sim", "rate"]].mean()
    ra = stats.linregress(g.sim, g.rate)
    ordd = tt.assign(sim_rank=tt.sim.fillna(-np.inf)).sort_values(["subject", "sim_rank"])
    ordd["pos"] = ordd.groupby("subject").cumcount()
    go = ordd.groupby("pos")[["sim", "rate"]].mean().dropna()
    rb = stats.linregress(go.sim, go.rate) if len(go) >= 3 else None
    rs_ = []
    for s, d in tt.dropna(subset=["sim"]).groupby("subject"):
        if len(d) >= 3 and d.rate.std() > 0 and d.sim.std() > 0:
            rs_.append(stats.pearsonr(d.sim, d.rate).statistic)
    rs_ = np.array(rs_)
    alt = "less" if metric == "corr" else "greater"
    tc, pc = stats.ttest_1samp(rs_, 0, alternative=alt)
    _, pc2 = stats.ttest_1samp(rs_, 0)
    # mixed model on similarity standardised over the table (raw distances are in the thousands)
    d = tt.dropna(subset=["sim"]).assign(sim=lambda x: (x.sim - x.sim.mean()) / x.sim.std())
    try:
        m = smf.mixedlm("rate ~ sim", d, groups=d["subject"], re_formula="~sim").fit()
        ok = bool(m.converged)
    except np.linalg.LinAlgError:
        ok = False
    if not ok:
        m = smf.mixedlm("rate ~ sim", d, groups=d["subject"]).fit()
    chi2, _, pm = wald(m, ["sim"])
    return ra, rb, rs_, tc, pc, pc2, chi2, pm, m.params["sim"]



FIG5 = {}
for metric in ("corr", "dist"):
    for prep in (("raw", "voxel") if metric == "corr" else ("raw", "voxel", "map")):
        for err_reading in ("span", "strict"):
            tt = transition_table(err_reading, prep, metric)
            ra, rb, rs_, tc, pc, pc2, chi2, pm, bm = fig5_tests(tt, metric)
            FIG5[(metric, prep, err_reading)] = (ra, rb, rs_, tc, pc)
            print(f"{metric}, maps {prep}, errors {err_reading} ({int(tt.rate.gt(0).sum())} participant-transitions with errors, "
                  f"{tt.sim.isna().sum()} NaN of {len(tt)}):\n"
                  f"   (a) types slope {ra.slope:.4g}, t6 = {ra.slope / ra.stderr:.2f}, p = {ra.pvalue:.3g} | (b) ordinal "
                  + (f"slope {rb.slope:.4g}, t = {rb.slope / rb.stderr:.2f}, p = {rb.pvalue:.3g}" if rb else "not computable (too few positions with a correlation)") + "\n"
                  f"   5C: mean within-participant r {rs_.mean():.3f} (n = {len(rs_)}), t = {tc:.2f}, one-sided p = {pc:.3g}, two-sided p = {pc2:.3g} | "
                  f"mixed model slope {bm:.4g} per SD, chi2(1) = {chi2:.2f}, p = {pm:.3g}")
# Control for trial counts (variant): errors take trials out of a state's regressor, so a state with
# more errors has fewer correct trials and noisier beta maps. Within participant, the correlation of
# distance with the Enter state's correct-trial count, and the partial correlation of distance and
# error rate with that count held constant.
for prep in ("raw", "voxel", "map"):
    tt = transition_table("span", prep, "dist")
    r_n, r_p = [], []
    for s, d in tt.groupby("subject"):
        r_n.append(stats.pearsonr(d.sim, d.n_ok).statistic)
        if d.rate.std() > 0:
            X = np.column_stack([np.ones(len(d)), d.n_ok])
            res = lambda v: v - X @ np.linalg.lstsq(X, v, rcond=None)[0]
            r_p.append(stats.pearsonr(res(d.sim.values), res(d.rate.values)).statistic)
    r_n, r_p = np.array(r_n), np.array(r_p)
    print(f"dist, maps {prep}, span errors: r(distance, correct trials) mean {r_n.mean():.3f}, two-sided p = {stats.ttest_1samp(r_n, 0).pvalue:.3g} | "
          f"partial r(distance, error rate | correct trials) mean {r_p.mean():.3f} (n = {len(r_p)}), one-sided p = {stats.ttest_1samp(r_p, 0, alternative='greater').pvalue:.3g}")
print("paper, correlations: t6 = -3.23, p = 0.02; Figure 5B p = 0.04; Figure 5C p < 0.01; mixed model chi2(1) = 3.0, p = 0.08")
print("paper, Euclidean: Figure S5B p = 0.03, Figure S5C p < 0.01")

**Result: not reproduced.** No reading gives a negative regression of error rate on pattern correlation
at p < 0.05: over the eight transition types the slope is negative but not significant (p = 0.08 to
0.37), and over ordinal positions it is positive where it reaches p < 0.05 (strict errors, raw maps
p = 0.049, voxel z-scoring p = 0.039). The within-participant correlations are not below 0 (mean r
-0.010 to 0.157, one-sided p >= 0.46), and the mixed model finds no effect (p >= 0.24; paper p = 0.08). Error rates rest
on the reconstructed Enter errors only.

**Result: reproduced, resting on the span reading of error states.** With Euclidean distances and the
span reading, error rate rises with distance over the ordinal positions (raw maps t = 6.17, p = 0.0008;
voxel z-scoring t = 4.10, p = 0.0064) and the within-participant correlations are above 0 (mean r 0.260,
p = 0.003; 0.313, p = 0.0003). With the strict reading (39 errors in 35 participant-transitions, 16 participants) neither holds at
p < 0.05 for raw maps, and with maps z-scored by map the relation is absent or reversed. The regression
over transition types is not significant in any reading (p >= 0.33). The paper reports the same result
for correlations, which is not found here (fig5-similarity-predicts-errors).

Errors take trials out of a state's regressor, so a state with more errors has fewer correct trials
and noisier maps. Distance falls with the Enter state's correct-trial count (mean within-participant
r -0.212, p = 0.013 raw; -0.258, p = 0.003 voxel z-scored), and with that count held constant the
distance and the error rate are not related (mean partial r 0.072, one-sided p = 0.22; 0.076,
p = 0.18). The paper's analysis has the same coupling, as its error trials were also left out of the
state regressors, so the outcome follows the criterion; the result may come from trial counts.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| behavior-error-rate | blocked: metadata insufficient (errors not countable) |
| supp-errors-by-trial-class | partially reproduced (assumption: one error per block) |
| supp-errors-stable-over-blocks | reproduced (assumption: one error per block) |
| supp-errors-not-by-age-category | reproduced (assumption: span reading) |
| supp-training-errors-higher | blocked: data missing |
| supp-training-rt-unchanged | blocked: data missing |
| supp-rt-stable-over-blocks | reproduced |
| supp-rt-faces-faster | reproduced |
| supp-rt-by-trial-class | reproduced |
| fig2a-16way-decoding | partially reproduced |
| fig2-two-back-state-space | partially reproduced |
| fig2b-hidden-components | partially reproduced |
| fig2b-current-age-absent | reproduced |
| fig2b-two-back-absent | reproduced |
| figs2c-8way-decoding | reproduced (assumption: voxel z-scoring) |
| figs2a-permutation-control | partially reproduced |
| synthetic-data-control | blocked: metadata insufficient |
| fig2c-enter-cluster | reproduced |
| fig2d-enter-nonenter-decoding | not reproduced |
| fig3-searchlight-conjunction | blocked: data missing |
| other-rois-incomplete | blocked: data missing |
| fig4a-decoding-error-correlation | partially reproduced (assumption: voxel z-scoring) |
| fig4b-error-trial-drop | not reproduced |
| fig4b-decline-before-errors | not reproduced |
| fig4b-error-vs-matched | not reproduced (assumption: class-matched comparison) |
| fig5-similarity-predicts-errors | not reproduced |
| figs5-euclidean-distance | reproduced (assumption: span reading) |
| fig5-synthetic-control | blocked: metadata insufficient |
| supp-motor-action-not-in-ofc | blocked: data missing |

How the analysis differs from the paper's:

- 24 of the 27 participants (those of the paper's trial-wise analysis, as the data set says).
- No spatial smoothing of beta maps or volumes: the files hold no voxel coordinates.
- The GLM's nuisance set is the data set's (four drift terms, six motion parameters), without the 26
  cardiac regressors of the paper; it uses the data set's design matrices, built with AFNI.
- Beta maps are z-scored per voxel across the state maps of a run (an assumption; the per-map
  reading is shown throughout).
- Errors are the error blocks of the event files, one per block, a lower bound; the state of an error
  trial is reconstructed from its neighbours (strict and span readings), and only Enter errors can be
  reconstructed.
- The logistic readings of the behavioural models use participant fixed effects, as statsmodels has no
  frequentist logistic mixed model; the linear mixed models use statsmodels' MixedLM in place of lmer.
- The data set's 'err' column puts the trials inside an error block after the error trial in no
  regressor; both readings of O3 are rebuilt from the error blocks and shown for the 16-way decoding
  and its components, with the same outcomes.
- Every error trial that can be given a state is an Enter trial, and with maps z-scored by voxel Enter
  trials decode worse than others on single volumes, so the error trial is compared with Enter trials of the matched courses
  (an assumption; the criterion's readings are shown).
- The trial-wise mixed models fall back to a random intercept where the random-slope fit did not converge.

What was found about the data set: the error events merge each error with its repetitions and often
the trials after it, so the number of errors, time-outs, error RTs and the state of most error trials
cannot be recovered; Enter errors are 181 of 228 error blocks, 79% against the 48% the paper's
class rates imply; the 'err' column of the design matrices
is not the boxcar the descriptor describes; three runs lack a 'first' column; one pair of adjacent
labelled trials in Subj-01 run 4 breaks the task rules; and the files hold no voxel coordinates, no
training session, no whole-brain data or other ROIs, and no response side or mapping.

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")